# Prerequisites
본 `ipynb` 은 `Python=3.12` 에서 작성하였습니다. Package dependency 를 해결하기 위해 아래 cell 을 실행해주세요.

## Install Python packages

In [ ]:
%pip -q install -U agent-framework==1.0.0b260130

## Load environment variables from a .env file
secret 노출을 피하고 notebook 들간의 일관된 환경변수를 설정하기 위해 `dotenv` 을 이용한다.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)

AZURE_MS_FOUNDRY_PROJECT_ENDPOINT = os.getenv("AZURE_MS_FOUNDRY_PROJECT_ENDPOINT")
AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
AZURE_OPENAI_CHAT_DEPLOYMENT = os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT")
AZURE_OPENAI_EMBEDDING_DEPLOYMENT = os.getenv("AZURE_OPENAI_EMBEDDING_DEPLOYMENT")

# Agentic Loop
Claude Code 공식 문서에 따르면:
> "When you give Claude a task, it works through three phases: gather context, take action, and verify results."
from IPython.display import Image, display

## ![agentic-loop](resources/agentic-loop.svg)
**핵심 구성 요소:**
- **Models**: 코드를 이해하고 작업을 추론
- **Tools**: 실제 액션을 수행 (파일 읽기/쓰기, 명령 실행 등)  
- **Agentic Harness**: 도구, 컨텍스트 관리, 실행 환경을 제공

> "Tools are what make Claude Code agentic. Without tools, Claude can only respond with text. With tools, Claude can act."

## ChatAgent 사용 (High-level)

Microsoft Agent Framework `ChatAgent` 이나 다른 Agent Packages 는 Agentic Loop를 내부적으로 처리합니다. 도구 호출과 결과 처리가 자동으로 수행됩니다.

In [ ]:
from pathlib import Path
from datetime import datetime
from typing import Annotated
from pydantic import Field
from agent_framework import tool

@tool(name="get_current_time", description="현재 시간을 반환합니다.")
def get_current_time() -> str:
    return datetime.now().strftime("%Y년 %m월 %d일 %H시 %M분 %S초")

@tool(name="calculate", description="수학 계산을 수행합니다. 사칙연산과 괄호를 지원합니다.")
def calculate(
    expression: Annotated[str, Field(description="계산할 수식 (예: '2 + 3 * 4')")]
) -> str:
    try:
        allowed_chars = set("0123456789+-*/.() ")
        if not all(c in allowed_chars for c in expression):
            return "❌ 오류: 허용되지 않은 문자가 포함되어 있습니다."
        return str(eval(expression))
    except Exception as e:
        return f"❌ 계산 오류: {str(e)}"

# 도구 목록
tools = [get_current_time, calculate]
print("✅ 도구 정의 완료:", [t.name for t in tools])

위에서 정의한 파일 시스템 도구들을 사용하는 에이전트를 만들어보자.

In [ ]:
from agent_framework import ChatAgent
from agent_framework.openai import OpenAIResponsesClient

# ChatAgent 생성
agent = ChatAgent(
    chat_client=OpenAIResponsesClient(
        base_url=AZURE_OPENAI_ENDPOINT,
        api_key=AZURE_OPENAI_API_KEY,
        model_id=AZURE_OPENAI_CHAT_DEPLOYMENT,
    ),
    name="AgenticLoopAgent",
    instructions="""당신은 도움이 되는 AI 어시스턴트입니다.
사용자의 요청을 수행하기 위해 필요한 도구를 사용할 수 있습니다.
도구를 사용할 때는 신중하게 판단하고, 결과를 명확하게 설명해주세요.
모든 파일 경로는 workspace 디렉토리 기준 상대 경로입니다.""",
    tools=tools,
)

Agent 의 응답을 출력하는 함수를 정의합니다.

In [ ]:
from agent_framework import Role

def print_result(result):
    """에이전트 실행 결과를 포맷팅하여 출력"""
    
    # 도구 호출 정보 출력
    print("\n======= 🤖 Agent Messages =======")
    for idx,msg in enumerate(result.messages):
        if msg.role == Role.ASSISTANT:
            for content in msg.contents:
                if content.type == "text":
                    print(f"[{idx}] 💬 {content.text}")
                elif content.type == "function_call":
                    print(f"[{idx}] 🔧 {content.type} | {content.name} | {content.raw_representation.arguments}")
                else:
                    print(f"[{idx}] not supported content type: {content.type}")
        elif msg.role == Role.TOOL:
            for content in msg.contents:
                print(f"[{idx}] 📤 {content.result}")
        else:
            print(f"[{idx}] not supported role: {msg.role}")

### 테스트: ChatAgent로 Agentic Loop 실행

`ChatAgent.run()`을 호출하면 내부적으로 Agentic Loop가 동작합니다.

In [ ]:
# 테스트 1: 단순 질문 (도구 미사용)
thread = agent.get_new_thread()

result = await agent.run("안녕하세요! 자기소개 해주세요.", thread=thread)
print_result(result)

In [ ]:
# 테스트 2: 도구 사용 (시간 조회)
result = await agent.run("현재 시간이 몇 시인지 알려주세요.", thread=thread)
print_result(result)

In [ ]:
# 테스트 3: 도구 사용 (수학 계산)
result = await agent.run("(123 + 456) * 2 를 계산해주세요.", thread=thread)
print_result(result)

## Agentic Loop 직접 구현 (Low-level)

`ChatAgent`가 내부적으로 어떻게 동작하는지 이해하기 위해 Agentic Loop를 직접 구현해봅니다.

In [ ]:
from openai import OpenAI
import json

# Azure OpenAI 클라이언트 생성
openai_client = OpenAI(
    base_url=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
)

# 도구 스키마 정의 (OpenAI Function Calling 형식)
TOOLS_SCHEMA = [
    {
        "type": "function",
        "function": {
            "name": "get_current_time",
            "description": "현재 시간을 반환합니다.",
            "parameters": {"type": "object", "properties": {}, "required": []}
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calculate",
            "description": "수학 계산을 수행합니다. 사칙연산과 괄호를 지원합니다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {"type": "string", "description": "계산할 수식"}
                },
                "required": ["expression"]
            }
        }
    }
]

# 도구 실행 함수 매핑
TOOL_FUNCTIONS = {
    "get_current_time": lambda: datetime.now().strftime("%Y년 %m월 %d일 %H시 %M분 %S초"),
    "calculate": lambda expression: str(eval(expression)) if all(c in "0123456789+-*/.() " for c in expression) else "오류",
}

print("✅ 도구 스키마 및 실행 함수 정의 완료!")

Agentic Loop 를 위한 함수를 구현해봅니다.

In [ ]:
def agentic_loop(user_message: str, max_iterations: int = 10) -> str:
    print(f"\n{'='*60}")
    print(f"🧑 사용자: {user_message}")
    print(f"{'='*60}")
    
    # 메시지 히스토리 초기화
    messages = [
        {"role": "system", "content": "당신은 도움이 되는 AI 어시스턴트입니다. 필요한 도구를 사용하여 사용자의 요청을 수행하세요."},
        {"role": "user", "content": user_message}
    ]

    # Agentic Loop 시작
    iteration = 0
    while iteration < max_iterations:
        iteration += 1
        print(f"\n--- 반복 {iteration} ---")
        
        # Step 1: LLM API 호출
        response = openai_client.chat.completions.create(
            model=AZURE_OPENAI_CHAT_DEPLOYMENT,
            messages=messages,
            tools=TOOLS_SCHEMA,
            tool_choice="auto"
        )
        
        assistant_message = response.choices[0].message
        finish_reason = response.choices[0].finish_reason
        print(f"📊 finish_reason: {finish_reason}")
        
        # Step 2: 도구 호출이 있는지 확인
        if assistant_message.tool_calls:
            print(f"🔧 도구 호출 감지: {len(assistant_message.tool_calls)}개")
            
            # 어시스턴트 메시지 저장 (도구 호출 포함)
            messages.append(assistant_message)
            
            # 각 도구 실행
            for tool_call in assistant_message.tool_calls:
                tool_name = tool_call.function.name
                arguments = json.loads(tool_call.function.arguments)
                
                print(f"  📌 실행: {tool_name}")
                print(f"     인자: {arguments}")
                
                # 도구 실행
                try:
                    result = TOOL_FUNCTIONS[tool_name](**arguments)
                except Exception as e:
                    result = f"오류: {str(e)}"
                
                print(f"     결과: {result[:100]}..." if len(str(result)) > 100 else f"     결과: {result}")
                
                # 도구 결과를 메시지에 추가
                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": str(result)
                })
            
            # 다음 반복으로 (도구 결과를 LLM에게 전달)
            continue
        
        else:
            # 텍스트 응답 - 루프 종료
            final_response = assistant_message.content
            print(f"\n🤖 AI 응답:\n{final_response}")
            return final_response
    
    # 최대 반복 횟수 초과
    return "❌ 오류: 최대 반복 횟수를 초과했습니다."

### 테스트: Low-level Agentic Loop

In [ ]:
# 테스트 2: 도구 사용 (시간 조회)
agentic_loop("현재 시간이 몇 시인지 알려주세요.")

In [ ]:
# 테스트 2: 복합 작업 
agentic_loop("현재 시간의 숫자들을 다 더해줘.")

## Wrap Up

### High-level vs Low-level 비교

| 항목 | ChatAgent (High-level) | 직접 구현 (Low-level) |
|------|------------------------|----------------------|
| Agentic Loop | `agent.run()` 내부 처리 | `while` 루프 직접 구현 |
| 도구 정의 | `@tool` 데코레이터 | JSON 스키마 + 함수 매핑 |
| 도구 실행 | 자동 처리 | `tool_calls` 파싱 후 수동 실행 |
| 메시지 관리 | `thread` 객체 | `messages` 리스트 직접 관리 |